# bansal
# CrimeSync Financial Forensics: Temporal Graph Neural Network for Money Laundering & Mule Ring Detection
### Temporal Dynamic GNN & Cycle Detection (Johnson's Algorithm) for UPI/IMPS Layering Trails

In [ ]:
# bansal
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import networkx as nx
from collections import defaultdict

print("[+] Initializing CrimeSync Temporal Financial Graph Neural Network (T-GNN)...")

### 1. Dynamic Transaction Stream Simulator with Smurfing & Layering Patterns

In [ ]:
# bansal
class FinancialStreamDataset:
    """Generates high-velocity banking transactions with rapid fan-out (layering) and fan-in (aggregation)"""
    def __init__(self, num_nodes=2500, num_tx=8000):
        self.num_nodes = num_nodes
        # Node features: [avg_balance, velocity_1h, rapid_in_out_ratio, kyc_risk, merchant_flag]
        self.node_features = torch.randn(num_nodes, 5)
        
        # Transaction edges: [src, dst, amount, timestamp, payment_rail]
        src = np.random.randint(0, num_nodes, size=num_tx)
        dst = np.random.randint(0, num_nodes, size=num_tx)
        amounts = np.random.exponential(scale=25000, size=num_tx)
        timestamps = np.sort(np.random.uniform(0, 86400, size=num_tx))
        
        self.df_tx = pd.DataFrame({
            'src': src,
            'dst': dst,
            'amount': amounts,
            'timestamp': timestamps,
            'rail': np.random.choice(['UPI', 'IMPS', 'RTGS', 'CRYPTO_ONRAMP'], size=num_tx)
        })
        
    def build_directed_multigraph(self):
        G = nx.MultiDiGraph()
        for _, row in self.df_tx.iterrows():
            G.add_edge(int(row['src']), int(row['dst']), amount=row['amount'], timestamp=row['timestamp'])
        return G

stream = FinancialStreamDataset()
G = stream.build_directed_multigraph()
print(f"[+] Financial Transaction Graph Built: {G.number_of_nodes()} accounts, {G.number_of_edges()} transaction hops")

### 2. Time-Aware Recurrent Graph Attention Layer (Temporal-GAT)

In [ ]:
# bansal
class TemporalGATCell(nn.Module):
    """Calculates time-decay weighted attention over incoming and outgoing financial transactions"""
    def __init__(self, in_features=5, time_dim=8, out_features=32):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.w_node = nn.Linear(in_features, out_features, bias=False)
        self.w_time = nn.Linear(1, time_dim, bias=False)
        self.attn_vector = nn.Parameter(torch.Tensor(1, out_features + time_dim))
        nn.init.xavier_uniform_(self.attn_vector)
        
    def forward(self, x, edge_index, edge_time):
        src, dst = edge_index[0], edge_index[1]
        h = self.w_node(x)
        t_emb = F.relu(self.w_time(edge_time.unsqueeze(-1)))
        
        # Edge representations
        edge_h = torch.cat([h[src], t_emb], dim=-1)
        attn_scores = F.leaky_relu((edge_h * self.attn_vector).sum(dim=-1))
        attn_weights = torch.softmax(attn_scores, dim=0)
        
        out = torch.zeros(x.size(0), self.out_features, device=x.device)
        out.index_add_(0, dst, h[src] * attn_weights.unsqueeze(-1))
        return F.elu(out)

class MuleRingDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.t_gat1 = TemporalGATCell(in_features=5, time_dim=8, out_features=32)
        self.t_gat2 = TemporalGATCell(in_features=32, time_dim=8, out_features=16)
        self.classifier = nn.Sequential(
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x, edge_index, edge_time):
        h1 = self.t_gat1(x, edge_index, edge_time)
        h2 = self.t_gat2(h1, edge_index, edge_time)
        return self.classifier(h2)

detector = MuleRingDetector()
print(detector)

### 3. Circular Money Laundering Cycle Identification (Johnson's Cycle Algorithm)

In [ ]:
# bansal
def extract_suspicious_layering_cycles(df_tx, min_cycle_len=3, max_cycle_len=6):
    """Extracts closed transaction loops indicative of layering and crypto off-ramping"""
    sub_G = nx.DiGraph()
    for _, row in df_tx.head(1000).iterrows():
        sub_G.add_edge(int(row['src']), int(row['dst']), weight=row['amount'])
        
    simple_cycles = list(nx.simple_cycles(sub_G))
    flagged_rings = [c for c in simple_cycles if min_cycle_len <= len(c) <= max_cycle_len]
    
    print(f"[!] Forensic Audit Discovered {len(flagged_rings)} Circular Layering Sub-Rings!")
    for i, ring in enumerate(flagged_rings[:3]):
        print(f"    Layering Ring #{i+1}: {' -> '.join(map(str, ring))} -> {ring[0]}")
    return flagged_rings

rings = extract_suspicious_layering_cycles(stream.df_tx)